In [15]:
# ========================================================
#  COMPONENT 4 - RAG Chain (Retrieval + Generation)
# ========================================================

from langchain_core.runnables import RunnableLambda

SMART_CONTRACT_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert legal and technical analyst specialising in blockchain smart contracts and legal documents.
Provide accurate, clear answers about contract terms, tokenomics, security, legal provisions, and employment terms.

CONTEXT FROM UPLOADED CONTRACT:
{context}

USER QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the provided contract context above.
- If information is absent, state: "This information is not found in the uploaded contract."
- Be precise with numbers, percentages, timeframes, and legal terms.
- Reference the relevant Article, Section, or Clause in your answer where possible.
- Use bullet points or numbered lists for multi-part answers.
- Do NOT speculate beyond what the contract states.

ANSWER:"""
)


class SmartContractRAGChain:
    """Full RAG pipeline: Query → Guardrails → Retrieval → Gemini → Citations"""

    def __init__(
        self,
        vector_store_manager: VectorStoreManager,
        guardrails: SmartContractGuardrails,
        top_k: int = 5,
        temperature: float = 0.1,
    ):
        self.vsm        = vector_store_manager
        self.guardrails = guardrails
        self.top_k      = top_k
        self.llm        = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash-lite",
            google_api_key=GEMINI_API_KEY,
            temperature=temperature,
            max_output_tokens=2048,
            convert_system_message_to_human=True,
        )

    def _format_sources(self, docs: List[Document]) -> str:
        """Format source citations as clean, readable numbered cards."""
        seen, out = set(), []
        for i, doc in enumerate(docs, 1):
            src  = doc.metadata.get("source", "Unknown")
            cid  = doc.metadata.get("chunk_id", "?")
            # Collapse all whitespace noise from DOCX/PDF parsing
            snip = " ".join(doc.page_content.split())[:180]
            key  = f"{src}_{cid}"
            if key not in seen:
                seen.add(key)
                out.append(
                    f"**[{i}] {src}** · chunk {cid}\n"
                    f"{snip}..."
                )
        return "\n\n".join(out)

    def ask(self, query: str) -> Tuple[str, str]:
        """Process a question through the full RAG pipeline. Returns (answer, sources)."""
        # Step 1: Input guardrails
        ok, reason = self.guardrails.validate_input(query)
        if not ok:
            return f"🚫 **Input blocked:** {reason}", ""

        # Step 2: Retrieve relevant chunks
        try:
            retriever = self.vsm.get_retriever(top_k=self.top_k)
        except RuntimeError as e:
            return f"⚠️ {e}", ""

        source_docs = retriever.invoke(query)
        if not source_docs:
            return "❌ No relevant content found. Try rephrasing your question.", ""

        # Step 3: Assemble context from retrieved chunks
        context = "\n\n---\n\n".join(d.page_content for d in source_docs)

        # Step 4: Generate answer via Gemini
        prompt   = SMART_CONTRACT_PROMPT.format(context=context, question=query)
        response = self.llm.invoke(prompt)
        raw      = response.content if hasattr(response, "content") else str(response)

        # Step 5: Output guardrails
        answer, warns = self.guardrails.validate_output(raw, query)
        if warns:
            answer += "\n\n" + "\n".join(warns)

        # Step 6: Format and return clean citations
        return answer, self._format_sources(source_docs)

    def as_runnable(self):
        """Expose the RAG chain as a LangChain Runnable for LangServe."""
        return RunnableLambda(lambda query: self.ask(query))


print("✅ SmartContractRAGChain defined.")

✅ SmartContractRAGChain defined.
